# Task 2: MySQL Database Pipeline – TechReads Retail Analytics
**Module:** CMP-X304-0 Data Engineering  
**Database:** techreads_db  
**Table:** books  
**Input:** techreads_books.csv

## Step 1: Import Required Libraries

In [5]:
import mysql.connector
import pandas as pd

## Step 2: Load and Inspect CSV Data

In [6]:
df = pd.read_csv('../Task 1/techreads_books.csv',
                 encoding='utf-8-sig',
                 na_values=['N/A', 'NA', 'n/a', ''])

print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
print('\nData types:')
print(df.dtypes)
print('\nFirst 5 rows:')
df.head()

Total rows: 15
Columns: ['Title', 'Author', 'Year', 'Rating', 'RatingsCount', 'ReviewsCount', 'Source', 'URL']

Data types:
Title            object
Author           object
Year              int64
Rating          float64
RatingsCount      int64
ReviewsCount      int64
Source           object
URL              object
dtype: object

First 5 rows:


,Title,Author,Year,Rating,RatingsCount,ReviewsCount,Source,URL
0,Fundamentals of Data Engineering: Plan and Bui...,Joe Reis,2022,4.18,927,97,goodreads,https://www.goodreads.com/book/show/61218623-f...
1,Designing Data-Intensive Applications,Martin Kleppmann,2015,4.70,10692,947,goodreads,https://www.goodreads.com/book/show/23463279-d...
2,The Data Warehouse Toolkit: The Complete Guide...,Ralph Kimball,1996,4.18,1036,65,goodreads,https://www.goodreads.com/book/show/748203.The...
3,Data Pipelines Pocket Reference: Moving and Pr...,James Densmore,2021,3.77,242,31,goodreads,https://www.goodreads.com/book/show/55841851-d...
4,Database Internals: A deep-dive into how distr...,Alex Petrov,2019,4.26,558,69,goodreads,https://www.goodreads.com/book/show/44647144-d...


## Step 3: Connect to MySQL and Create Database

In [7]:
conn = mysql.connector.connect(
    host='localhost',
    port=8868,
    user='root',
    password='123123'  
)
cursor = conn.cursor()

cursor.execute('CREATE DATABASE IF NOT EXISTS techreads_db')
print('Database techreads_db created (or already exists).')

cursor.execute('USE techreads_db')
print('Using database: techreads_db')

Database techreads_db created (or already exists).
Using database: techreads_db


## Step 4: Create the Books Table

In [8]:
cursor.execute('DROP TABLE IF EXISTS books')

create_table_query = '''
CREATE TABLE books (
    id             INT AUTO_INCREMENT PRIMARY KEY,
    title          VARCHAR(255) NOT NULL,
    author         VARCHAR(255),
    year           VARCHAR(10),
    rating         DECIMAL(3,2),
    ratings_count  INT,
    reviews_count  INT,
    source         VARCHAR(50),
    url            VARCHAR(500)
)
''';
cursor.execute(create_table_query)
print('Table books created successfully.')

cursor.execute('DESCRIBE books')
print('\nTable structure:')
for row in cursor.fetchall():
    print(row)

Table books created successfully.

Table structure:
('id', 'int', 'NO', 'PRI', None, 'auto_increment')
('title', 'varchar(255)', 'NO', '', None, '')
('author', 'varchar(255)', 'YES', '', None, '')
('year', 'varchar(10)', 'YES', '', None, '')
('rating', 'decimal(3,2)', 'YES', '', None, '')
('ratings_count', 'int', 'YES', '', None, '')
('reviews_count', 'int', 'YES', '', None, '')
('source', 'varchar(50)', 'YES', '', None, '')
('url', 'varchar(500)', 'YES', '', None, '')


## Step 5: Import CSV Data into MySQL

In [9]:
insert_query = '''
    INSERT INTO books (title, author, year, rating, ratings_count, reviews_count, source, url)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
''';

inserted = 0
for _, row in df.iterrows():
    title = row['Title'] if not pd.isna(row['Title']) else None
    author = row['Author'] if not pd.isna(row['Author']) else None
    year = str(int(row['Year'])) if not pd.isna(row['Year']) else None
    rating = float(row['Rating']) if not pd.isna(row['Rating']) else None
    ratings_count = int(row['RatingsCount']) if not pd.isna(row['RatingsCount']) else None
    reviews_count = int(row['ReviewsCount']) if not pd.isna(row['ReviewsCount']) else None
    source = row['Source'] if not pd.isna(row['Source']) else None
    url = row['URL'] if not pd.isna(row['URL']) else None
    
    cursor.execute(insert_query, (
        title,
        author,
        year,
        rating,
        ratings_count,
        reviews_count,
        source,
        url
    ))
    inserted += 1

conn.commit()
print(f'Successfully inserted {inserted} records into books table.')

Successfully inserted 15 records into books table.


## Step 6: SQL Queries

In [10]:
query1 = '''
    SELECT title, rating, ratings_count
    FROM books
    ORDER BY rating DESC
''';
cursor.execute(query1)
results1 = cursor.fetchall()

print('Query 1: All books ordered by rating (high to low)')
print(f'{"Title":<70} {"Rating":<10} {"RatingsCount"}')
print('-' * 95)
for row in results1:
    print(f'{str(row[0])[:67]:<70} {str(row[1]):<10} {row[2]}')

Query 1: All books ordered by rating (high to low)
Title                                                                  Rating     RatingsCount
-----------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                                  4.70       10692
Data Pipelines with Apache Airflow                                     4.31       118
Learning Spark: Lightning-Fast Data Analytics                          4.30       152
Database Internals: A deep-dive into how distributed data systems w    4.26       558
Making Sense of Stream Processing                                      4.25       190
Data Engineering with AWS: Learn how to design and build cloud-base    4.21       19
Architecting Modern Data Platforms: A Guide to Enterprise Hadoop at    4.19       26
Fundamentals of Data Engineering: Plan and Build Robust Data System    4.18       927
The Data Warehouse Toolkit: The Complete Guide to Dimensional Model   

In [11]:
query2 = '''
    SELECT title, author, rating, reviews_count
    FROM books
    WHERE rating >= 4.0
    ORDER BY rating DESC
''';
cursor.execute(query2)
results2 = cursor.fetchall()

print('Query 2: Books with rating >= 4.0')
print(f'{"Title":<55} {"Author":<20} {"Rating":<10} {"Reviews"}')
print('-' * 100)
for row in results2:
    print(f'{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<10} {row[3]}')

Query 2: Books with rating >= 4.0
Title                                                   Author               Rating     Reviews
----------------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                   Martin Kleppmann     4.70       947
Data Pipelines with Apache Airflow                      Bas P. Harenslak     4.31       16
Learning Spark: Lightning-Fast Data Analytics           Jules S. Damji       4.30       12
Database Internals: A deep-dive into how distributed    Alex Petrov          4.26       69
Making Sense of Stream Processing                       Martin Kleppmann     4.25       30
Data Engineering with AWS: Learn how to design and b    Gareth Eagar         4.21       2
Architecting Modern Data Platforms: A Guide to Enter    Jan Kunigk           4.19       8
Fundamentals of Data Engineering: Plan and Build Rob    Joe Reis             4.18       97
The Data Warehouse Toolkit: The Complete G

In [12]:
query3 = '''
    SELECT title, author, ratings_count, reviews_count
    FROM books
    ORDER BY ratings_count DESC
    LIMIT 5
''';
cursor.execute(query3)
results3 = cursor.fetchall()

print('Query 3: Top 5 most rated books')
print(f'{"Title":<55} {"Author":<20} {"Ratings":<10} {"Reviews"}')
print('-' * 100)
for row in results3:
    print(f'{str(row[0])[:52]:<55} {str(row[1])[:17]:<20} {str(row[2]):<10} {row[3]}')

Query 3: Top 5 most rated books
Title                                                   Author               Ratings    Reviews
----------------------------------------------------------------------------------------------------
Designing Data-Intensive Applications                   Martin Kleppmann     10692      947
The Data Warehouse Toolkit: The Complete Guide to Di    Ralph Kimball        1036       65
Fundamentals of Data Engineering: Plan and Build Rob    Joe Reis             927        97
Kafka: The Definitive Guide: Real-Time Data and Stre    Neha Narkhede        727        81
Database Internals: A deep-dive into how distributed    Alex Petrov          558        69


## Step 7: Close Connection

In [13]:
cursor.close()
conn.close()
print('MySQL connection closed.')

MySQL connection closed.
